# Portfolio Top Movers

A simple notebook that ranks stocks by price change, searches recent news with [Bigdata.com](https://bigdata.com), and uses an OpenAI ChatGPT model to summarize the most important developments.

Run the cells from top to bottom:

1. Install packages.
2. Load API keys from `.env`.
3. Edit the configuration.
4. Run the analysis and review the results.


## 1. Install packages

This notebook uses OpenAI only; no Google Gemini package or credentials are required.

In [10]:
%pip install -q aiohttp openai pandas pydantic python-dotenv pyyaml requests


/Users/bakulkumarkakadiya/dev/github/portfolio-top-movers/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## 2. Load environment variables

Create a `.env` file in the repository root:

```text
BIGDATA_API_KEY=your_bigdata_key
OPENAI_API_KEY=your_openai_key
OPENAI_MODEL=gpt-5-mini
```

`OPENAI_MODEL` is optional and defaults to `gpt-5-mini`.

In [11]:
import os

from dotenv import load_dotenv

load_dotenv()

BIGDATA_API_KEY = os.getenv("BIGDATA_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")
BIGDATA_BASE_URL = "https://api.bigdata.com/v1"

if not BIGDATA_API_KEY:
    raise ValueError("Add BIGDATA_API_KEY to your .env file.")
if not OPENAI_API_KEY:
    raise ValueError("Add OPENAI_API_KEY to your .env file.")

print(f"Environment loaded. OpenAI model: {OPENAI_MODEL}")

Environment loaded. OpenAI model: gpt-5-mini


## 3. Configuration

Edit this cell to change the ticker universe, lookback period, number of movers, model summaries, or news topics.

In [12]:
TICKERS: list[str] = [
    "AAPL",
    "AMZN",
    "TSLA",
    "NVDA",
    "AVGO",
    "INTC",
    "MSFT",
    "GOOGL",
    "META",
    "TSM",
    "NFLX",
]

TIME_PERIOD_DAYS = 1
TOP_N = 2
MAX_SUMMARY_BULLETS = 3

SEARCH_TOPICS: list[dict[str, str]] = [
    {
        "topic_name": "Earnings",
        "topic_text": "{company} earnings results revenue profit guidance",
    },
    {
        "topic_name": "Analyst",
        "topic_text": "{company} analyst rating upgrade downgrade price target",
    },
    {
        "topic_name": "Products",
        "topic_text": "{company} product launch feature release major update",
    },
    {
        "topic_name": "M&A",
        "topic_text": "{company} acquisition merger divestiture strategic partnership",
    },
]

print(f"{len(TICKERS)} tickers | {TIME_PERIOD_DAYS}-day window | top {TOP_N} each way")


11 tickers | 1-day window | top 2 each way


In [13]:
if TIME_PERIOD_DAYS <= 0:
    raise ValueError("TIME_PERIOD_DAYS must be greater than zero.")
if TOP_N <= 0:
    raise ValueError("TOP_N must be greater than zero.")

print("Topics:", ", ".join(topic["topic_name"] for topic in SEARCH_TOPICS))


Topics: Earnings, Analyst, Products, M&A


## 4. Imports and services

The OpenAI service is created explicitly, so the notebook cannot fall back to another LLM provider.

In [14]:
import asyncio
from typing import Any

import pandas as pd
import requests
from IPython.display import Markdown, display
from pydantic import BaseModel

from services.movers_workflow import MoverData, fetch_entity_ids_batch, fetch_news_for_mover
from services.openai_service import OpenAIService
from services.price_service import get_latest_price
from services.report_service import ReportService, TopicBrief
from services.topic_search_service import TopicSearchService

openai_service = OpenAIService(api_key=OPENAI_API_KEY, model=OPENAI_MODEL)
report_service = ReportService(llm_service=openai_service)

print(f"OpenAI ready: {openai_service.model}")

OpenAI ready: gpt-5-mini


## 5. Small helper functions

The API provides fixed price windows, so the requested lookback is mapped to the nearest available window. OpenAI then selects the most material summary bullets.


In [15]:
PRICE_WINDOWS: tuple[tuple[int, str], ...] = (
    (1, "1D"),
    (5, "5D"),
    (30, "1M"),
    (90, "3M"),
    (180, "6M"),
    (365, "1Y"),
)


def price_window(days: int) -> str:
    """Return the API price window closest to the requested number of days."""
    return min(PRICE_WINDOWS, key=lambda item: abs(item[0] - days))[1]


def fetch_price_change(entity_id: str | None, window: str) -> float | None:
    """Fetch an entity's percentage price change for one window."""
    if not entity_id:
        return None

    response = requests.post(
        f"{BIGDATA_BASE_URL}/price/changes/query",
        headers={"X-API-KEY": BIGDATA_API_KEY},
        json={"identifier": {"type": "rp_entity_id", "value": entity_id}},
        timeout=15,
    )
    response.raise_for_status()
    results: list[dict[str, Any]] = response.json().get("results", [])
    return results[0].get(window) if results else None


RANKING_WINDOW = price_window(TIME_PERIOD_DAYS)
print(f"Ranking on {RANKING_WINDOW}; searching {TIME_PERIOD_DAYS} day(s) of news.")

Ranking on 1D; searching 1 day(s) of news.


In [16]:
class SummaryBullet(BaseModel):
    """One ranked summary bullet returned by OpenAI."""

    rank: int
    topic_name: str
    bullet: str


async def summarize_briefs(
    briefs: list[TopicBrief],
    company_name: str,
) -> list[SummaryBullet]:
    """Select the most material briefs for one company."""
    if not briefs:
        return []

    candidates = "\n".join(
        f"- [{brief.topic_name}] {brief.bullet_point}" for brief in briefs
    )
    prompt = f"""You are an equity analyst summarizing news about {company_name}.

Select at most {MAX_SUMMARY_BULLETS} market-moving items from these candidates:
{candidates}

Rank them by materiality, keep each original topic name, and do not add outside facts.
"""
    bullets = await openai_service.generate_content_list(
        prompt=prompt,
        response_schema=SummaryBullet,
    )
    return sorted(bullets, key=lambda bullet: bullet.rank)[:MAX_SUMMARY_BULLETS]

## 6. Run the analysis

This resolves tickers, fetches prices, selects gainers and decliners, searches their recent news, and creates OpenAI summaries.


In [17]:
def normalize_tickers(tickers: list[str]) -> list[str]:
    """Normalize ticker symbols and remove duplicates."""
    normalized = (ticker.split(":")[-1].strip().upper() for ticker in tickers)
    return list(dict.fromkeys(ticker for ticker in normalized if ticker))


def rank_movers(movers: list[MoverData]) -> dict[str, list[MoverData]]:
    """Return the largest positive and negative movers."""
    valid = [mover for mover in movers if mover.price_change_pct is not None]
    gainers = sorted(valid, key=lambda mover: mover.price_change_pct or 0, reverse=True)
    decliners = sorted(valid, key=lambda mover: mover.price_change_pct or 0)
    return {
        "gainers": [mover for mover in gainers if (mover.price_change_pct or 0) > 0][:TOP_N],
        "decliners": [mover for mover in decliners if (mover.price_change_pct or 0) < 0][:TOP_N],
    }


async def analyze_mover(
    mover: MoverData,
    search_service: TopicSearchService,
) -> dict[str, Any]:
    """Search and summarize recent news for one mover."""
    mover = await fetch_news_for_mover(
        mover,
        search_service,
        days=TIME_PERIOD_DAYS,
        custom_topics=SEARCH_TOPICS,
    )
    articles: list[dict[str, Any]] = (mover.news_data or {}).get("topic_results", [])
    news = {
        "ticker": mover.ticker,
        "company_name": mover.company_name,
        "topic_results": articles,
    }
    briefs = await report_service.generate_topic_briefs(news) if articles else []
    bullets = await summarize_briefs(briefs, mover.company_name)
    return {"mover": mover, "articles": len(articles), "bullets": bullets}


async def run_analysis() -> dict[str, Any]:
    """Run the complete top-movers workflow."""
    search_service = TopicSearchService(BIGDATA_API_KEY, BIGDATA_BASE_URL)
    try:
        tickers = normalize_tickers(TICKERS)
        entities = await fetch_entity_ids_batch(tickers, search_service)

        movers: list[MoverData] = []
        for ticker in tickers:
            entity = entities[ticker]
            entity_id = entity.get("entity_id")
            price = (
                get_latest_price(entity_id, ticker, BIGDATA_API_KEY) or {}
                if entity_id
                else {}
            )
            movers.append(
                MoverData(
                    ticker=ticker,
                    company_name=entity.get("company_name", ticker),
                    entity_id=entity_id,
                    current_price=price.get("price"),
                    price_change_pct=fetch_price_change(entity_id, RANKING_WINDOW),
                    currency=price.get("currency", "USD"),
                )
            )

        ranked = rank_movers(movers)
        results: dict[str, Any] = {"window": RANKING_WINDOW}
        for group in ("gainers", "decliners"):
            results[group] = await asyncio.gather(
                *(analyze_mover(mover, search_service) for mover in ranked[group])
            )
        return results
    finally:
        await search_service.close()


RESULTS = await run_analysis()
print("Analysis complete.")


Analysis complete.


## 7. Results

The table shows the selected movers, followed by their most important news summaries.


In [18]:
def format_percent(value: object) -> str:
    """Format a percentage for display."""
    return f"{value:+.2f}%" if isinstance(value, (int, float)) else "N/A"


def format_price(value: object) -> str:
    """Format a price for display."""
    return f"{value:,.2f}" if isinstance(value, (int, float)) else "N/A"


rows: list[dict[str, Any]] = []
for group, label in (("gainers", "Gainer"), ("decliners", "Decliner")):
    for result in RESULTS[group]:
        mover = result["mover"]
        rows.append(
            {
                "Type": label,
                "Ticker": mover.ticker,
                "Company": mover.company_name,
                "Price": format_price(mover.current_price),
                f"{RESULTS['window']} Change": format_percent(mover.price_change_pct),
                "Articles": result["articles"],
            }
        )

display(pd.DataFrame(rows))

for group, heading in (("gainers", "Top Gainers"), ("decliners", "Top Decliners")):
    display(Markdown(f"## {heading}"))
    for result in RESULTS[group]:
        mover = result["mover"]
        display(
            Markdown(
                f"### {mover.company_name} ({mover.ticker}) — "
                f"{format_percent(mover.price_change_pct)}"
            )
        )
        if result["bullets"]:
            summary = "\n".join(
                f"{bullet.rank}. **[{bullet.topic_name}]** {bullet.bullet}"
                for bullet in result["bullets"]
            )
        else:
            summary = "_No significant news found for the selected period._"
        display(Markdown(summary))


,Type,Ticker,Company,Price,1D Change,Articles
0,Gainer,MSFT,Microsoft Corp.,500.17,+2.54%,40
1,Gainer,AVGO,Broadcom Inc.,420.58,+0.55%,14
2,Decliner,TSM,TSMC Ltd. (Taiwan),"2,375.00",-1.66%,45
3,Decliner,GOOGL,Alphabet Inc.,357.87,-1.29%,33


## Top Gainers

### Microsoft Corp. (MSFT) — +2.54%

1. **[Earnings]** Microsoft posted FY26 Q4 revenue of $90B; Azure grew 43% YoY (annual Azure revenue exceeded $100B), guided ~45% Azure growth next quarter, and generated ~$19.6B free cash flow — underscores continued cloud-driven revenue momentum and strong cash generation.
2. **[Products]** Microsoft opened its largest Indian data center ('South Central India') on Aug 6, 2026, expanding Azure to four India regions, signed early customers (Adani, HDFC Bank), and committed ~$20.5B in India expansion — expands local footprint and customer access in a key growth market.
3. **[M&A]** No significant M&A developments disclosed in the provided materials; no announced acquisitions, divestitures, or strategic partnerships impacting near-term financials as of 2026-08-06.

### Broadcom Inc. (AVGO) — +0.55%

...

## Top Decliners

### TSMC Ltd. (Taiwan) (TSM) — -1.66%

1. **[[Earnings] * **TSMC Ltd. (Taiwan)**]** Boosted 2026 capex to $60–64bn and raised USD revenue growth guidance to >+30% YoY; management guides 2Q26 revenue +10.3% QoQ with gross margin at ~67.5%.
2. **[[Products] * **TSMC Ltd. (Taiwan)**]** Accelerating 3nm capacity to ~180,000 wafers/month by early Q4 2026 and targeting combined 2nm+3nm ~250,000 wafers/month by year-end, materially expanding advanced-node shipment capacity for AI customers.
3. **[[Analyst] * **TSMC Ltd. (Taiwan)**]** CRISPidea raised FY27 target price to NT$2,650 (from NT$2,330), maintained Buy, implying ~12.8% upside versus NT$2,350 CMP and FY27 adjusted EPS forecast NT$138.76.

### Alphabet Inc. (GOOGL) — -1.29%

1. **[Earnings]** Alphabet reported Q2 2026 revenue $119.8B (+24% YoY); Google Cloud $24.768B (+82% YoY); Q2 capex $44.9B and full‑year capex raised to $195–205B, resulting in Q2 free cash flow of -$5.9B (first negative FCF).
2. **[Products]** Demis Hassabis moved to DeepMind Chairman/Alphabet Chief Scientist (announced Aug 5, 2026); Korey Kafai promoted to run DeepMind/Gemini amid delays to Gemini 3.5 Pro and a push to accelerate productization.
3. **[Analyst]** No analyst rating changes reported as of 2026-08-06; Moody's flagged upgrade/downgrade triggers tied to sustained revenue/profit growth, free‑cash‑flow and capital policy—no rating action disclosed publicly.